## Plotting with SCAN Data

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
def plot_nutrient_data(df, start_date, end_date, param, ax, check_valid = True, **kwargs):
    df.loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, label = param, **kwargs)


def plot_nutrient_data_by_bottle(df, start_date, end_date, param, ax, check_valid = True, **kwargs):
    first_rep = df['Bottle Replicate'] == 1
    print(first_rep)
    df[first_rep].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, label = 'First Bottle',  **kwargs)
    df[~first_rep].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', color = 'r', rot=45, label = 'Second Bottle',  **kwargs)


def plot_nutrient_data_by_rundate(df, start_date, end_date, param, ax, check_valid=True, **kwargs):
    rundates = df['AA500 Run Date'].unique()
    from itertools import cycle
    color_cycler = plt.rcParams['axes.prop_cycle'].by_key()['color']
    colors = cycle(color_cycler)
    for date, color in zip(rundates, colors):
        df[df['AA500 Run Date'] == date].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, color=color, label = 'Run on: ' + date.strftime('%m/%d/%Y'),  **kwargs)
    

In [ ]:
result_dir = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/Processed Data/'
rme_results = pd.read_csv(result_dir+'rme_aa500.csv', index_col='Sample Datetime', keep_default_na=False, na_values='NaN', parse_dates=True)
rme_results

In [ ]:
rme_cleaned = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned.csv', index_col='Date/Time', parse_dates=True)
rme_cleaned

In [ ]:
fig, ax= plt.subplots(figsize=(15, 9), nrows=2, sharex=True)
plot_nutrient_data_by_bottle(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[0])
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], color=['g', 'y'], ax=ax[0], rot=45, title='All Data')

plot_nutrient_data_by_bottle(rme_results[rme_results['Nitrate QA']==''], '02/01/25', '11/01/25', 'Nitrate', ax[1])
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], color=['g', 'y'], ax=ax[1], rot=45, title='Excluding Flagged Data')

fig.tight_layout()

## Correlation Plots

In [ ]:
def fit_linear_function(linear_range, x, y):
    coeff = np.polyfit(linear_range[x], linear_range[y], 1)
    r2 = np.corrcoef(linear_range[x], linear_range[y])[0,1]**2
    return coeff, r2

def plot_linear_fit(data, linear_range, x, y, text_start_x, text_start_y, text_y_spacing, ax):
    coeff, r2 = fit_linear_function(linear_range, x, y)
    vals= np.polyval(coeff, data[x])
    ax.plot(data[x], vals, color='b')
    ax.text(text_start_x, text_start_y, 'Slope: %.4f' % (coeff[0]))
    ax.text(text_start_x, text_start_y - text_y_spacing, 'Intercept: %.4f' % (coeff[1]))
    ax.text(text_start_x, text_start_y - 2*text_y_spacing, 'R2: %.4f' % (r2))
    return coeff, r2

In [ ]:
rme_merged = pd.merge_asof(rme_results, rme_cleaned, left_index=True, right_index=True, direction='nearest', tolerance = pd.Timedelta('1h'))
rme_merged_clean = rme_merged.drop(['2/15/25 3:30:00', '2-22-25 11:00']).dropna(subset=['Nitrate mean', 'two_wavelength_no3_mgl_correct']) # drop two outliers from timeseries


In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(11,8.5))

rme_merged_clean.plot(x='Nitrate mean', y='one_wavelength_no3_mgl', kind='scatter', xerr='Nitrate err', ax=ax[0,0], title='Uncorrected One Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'one_wavelength_no3_mgl', .3, .2, .02, ax[0,0])

rme_merged_clean.plot(x='Nitrate mean', y='one_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,0], title='Corrected One Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'one_wavelength_no3_mgl_correct', .3, 2, .3, ax[1,0])

rme_merged_clean.plot(x='Nitrate mean', y='two_wavelength_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[0,1], title='Uncorrected Two Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'two_wavelength_no3_mgl', .3, .15, .02, ax[0,1])

rme_merged_clean.plot(x='Nitrate mean', y='two_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,1], title='Corrected Two Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'two_wavelength_no3_mgl_correct', .3, .2, .03, ax[1,1])

rme_merged_clean.plot(x='Nitrate mean', y='second_derivative_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[0,2], title='Uncorrected Second Derivative')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl', .3, .3, .03, ax[0,2])

rme_merged_clean.plot(x='Nitrate mean', y='second_derivative_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,2], title='Corrected Second Derivative')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl_correct', .3, .3, .03, ax[1,2])


fig.suptitle('RME Calibration Plots')
fig.tight_layout()


## Plotting Other Nutrients

In [ ]:
rme_results_copy = rme_results
rme_results_copy['N-P Ratio mean'] = rme_results_copy['Nitrate mean']/rme_results_copy['Phosphate mean']
rme_results_copy['N-NH3 Ratio mean'] = rme_results_copy['Nitrate mean']/rme_results_copy['Ammonium mean']
rme_results_copy['N-P Ratio err'] = 0
rme_results_copy['N-NH3 Ratio err'] = 0


In [ ]:
fig, ax= plt.subplots(figsize=(15, 9),nrows=2,  sharex=True)
plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'Nitrate', ax[0], color='green', title= 'Concentration')
plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'Phosphate',  ax[0],color='red')
plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'Ammonium', ax[0], color='blue' )

plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'N-P Ratio', ax[1], color='red', title= 'Ratio', logy=True)
plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'N-NH3 Ratio', ax[1], color='blue', title= 'Ratio', logy=True)

ax[0].set_ylabel('Concentration (mg/L)')
ax[1].set_ylabel('Concentration Ratio')
ax[1].axhline(1, color='k', ls='--')


fig.tight_layout()

In [ ]:
fieldblank_results = pd.read_csv(result_dir+'field_blank_aa500.csv', index_col='Sample Datetime', keep_default_na=False, na_values='NaN', parse_dates=True)